# Feed Forward Network (FFN) in combination with Multi-Head Attention (MHA)

## MHA and FFN Modules:

In [1]:
import torch
import torch.nn as nn
import math

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        # d_k is CALCULATED, not chosen by us
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # each head's dimension

        # Each head needs its own Q, K, V projections
        # But instead of making separate nn.Linear for each head,
        # we use ONE big linear layer and then SPLIT the output
        # This is more efficient — one big matrix multiply instead of many small ones
        self.W_Q = nn.Linear(d_model, d_model, bias=False)  # (8, 8) not (8, 4)!
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)

        # After concatenating all heads, project back to d_model
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        seq_len = x.shape[0]     # number of tokens (7)

        # Step 1: Project into Q, K, V (full d_model size)
        Q = self.W_Q(x)   # (7, 8)
        K = self.W_K(x)   # (7, 8)
        V = self.W_V(x)   # (7, 8)

        # Step 2: SPLIT into multiple heads
        # Reshape from (7, 8) → (7, 2, 4) → (2, 7, 4)
        # 7 tokens, 2 heads, 4 dimensions per head
        Q = Q.view(seq_len, self.num_heads, self.d_k).permute(1, 0, 2)
        K = K.view(seq_len, self.num_heads, self.d_k).permute(1, 0, 2)
        V = V.view(seq_len, self.num_heads, self.d_k).permute(1, 0, 2)
        # Now Q, K, V are each (2, 7, 4) — 2 heads, 7 tokens, 4 dims per head

        # Step 3: Attention (same formula, now per head)
        scores = Q @ K.transpose(-2, -1)       # (2, 7, 7) — each head has its own 7×7
        scaled_scores = scores / math.sqrt(self.d_k)
        attention_weights = torch.softmax(scaled_scores, dim=-1)  # (2, 7, 7)

        # Step 4: Multiply by V
        head_outputs = attention_weights @ V    # (2, 7, 4)

        # Step 5: CONCATENATE all heads back together
        # (2, 7, 4) → (7, 2, 4) → (7, 8)
        head_outputs = head_outputs.permute(1, 0, 2).contiguous().view(seq_len, self.d_model)

        # Step 6: Final projection
        output = self.W_O(head_outputs)    # (7, 8)

        return output, attention_weights


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        # Two linear layers — expand then compress
        # nn.Linear handles weights AND biases automatically
        self.layer1 = nn.Linear(d_model, d_ff)       # (8 → 32) expand
        self.relu = nn.ReLU()                        # activation
        self.layer2 = nn.Linear(d_ff, d_model)       # (32 → 8) compress

    def forward(self, x):
        # Each step on its own line for clarity
        x = self.layer1(x)    # expand:   (7, 8)  → (7, 32)
        x = self.relu(x)      # activate: (7, 32) → (7, 32) (negatives → 0)
        x = self.layer2(x)    # compress: (7, 32) → (7, 8)
        return x


In [2]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()

        # The two main components
        self.attention = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = FeedForward(d_model, d_ff)

        # Layer normalization (stabilizes training)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        # === ATTENTION BLOCK ===
        # 1. Normalize
        normed = self.norm1(x)
        # 2. Multi-head attention
        attn_output, attn_weights = self.attention(normed)
        # 3. Residual connection (add original input back)
        x = x + attn_output

        # === FFN BLOCK ===
        # 4. Normalize
        normed = self.norm2(x)
        # 5. Feed-forward
        ff_output = self.feed_forward(normed)
        # 6. Residual connection (add input back again)
        x = x + ff_output

        return x, attn_weights

Each sub-block follows: Normalize → Process → Add back original. This pattern repeats throughout the entire Transformer.

```
Input
  ↓
  ├──────────────────────┐
  ↓                      │
  LayerNorm              │  Residual
  ↓                      │  Connection
  Multi-Head Attention   │
  ↓                      │
  + ←────────────────────┘  (add original back)
  ↓
  ├──────────────────────┐
  ↓                      │
  LayerNorm              │  Residual
  ↓                      │  Connection
  Feed-Forward           │
  ↓                      │
  + ←────────────────────┘  (add original back)
  ↓
Output
```

### Run it:

In [3]:
# Full pipeline: Text → Tokens → Embeddings → Transformer Block

# Sentence setup
text = "I love code and I love AI"
tokens = text.lower().split()
vocab = sorted(set(tokens))
word_to_id = {word: idx for idx, word in enumerate(vocab)}
token_ids = torch.tensor([word_to_id[t] for t in tokens])

# Config
d_model = 8
num_heads = 2
d_ff = 32       # 4 × d_model

# Embedding + Positional Encoding
embedding = nn.Embedding(len(vocab), d_model)

# Create positional encoding
def create_positional_encoding(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    for pos in range(max_len):
        for i in range(0, d_model, 2):
            denominator = 10000 ** (i / d_model)
            pe[pos, i]     = math.sin(pos / denominator)
            pe[pos, i + 1] = math.cos(pos / denominator)
    return pe

pe = create_positional_encoding(10, d_model)
final_embeddings = embedding(token_ids) + pe[:len(tokens)]

# Create and run the Transformer block
transformer_block = TransformerBlock(d_model=8, num_heads=2, d_ff=32)
output, attention_weights = transformer_block(final_embeddings)

print("=== FULL TRANSFORMER BLOCK ===")
print(f"Input shape:  {final_embeddings.shape}")   # (7, 8)
print(f"Output shape: {output.shape}")              # (7, 8)
print(f"Attention:    {attention_weights.shape}")    # (2, 7, 7)

# Count ALL parameters
total = sum(p.numel() for p in transformer_block.parameters())
print(f"\nTotal trainable parameters: {total}")

print("\n=== Layer by layer breakdown ===")
for name, param in transformer_block.named_parameters():
    print(f"  {name}: {param.shape}")

=== FULL TRANSFORMER BLOCK ===
Input shape:  torch.Size([7, 8])
Output shape: torch.Size([7, 8])
Attention:    torch.Size([2, 7, 7])

Total trainable parameters: 840

=== Layer by layer breakdown ===
  attention.W_Q.weight: torch.Size([8, 8])
  attention.W_K.weight: torch.Size([8, 8])
  attention.W_V.weight: torch.Size([8, 8])
  attention.W_O.weight: torch.Size([8, 8])
  feed_forward.layer1.weight: torch.Size([32, 8])
  feed_forward.layer1.bias: torch.Size([32])
  feed_forward.layer2.weight: torch.Size([8, 32])
  feed_forward.layer2.bias: torch.Size([8])
  norm1.weight: torch.Size([8])
  norm1.bias: torch.Size([8])
  norm2.weight: torch.Size([8])
  norm2.bias: torch.Size([8])


## Flow of Execution:

```
INPUT
"I love code and I love AI"
token_ids = [3, 4, 2, 1, 3, 4, 0]
shape: (7,)  ← just 7 integers
        │
        ▼
┌─────────────────────────────┐
│     TOKEN EMBEDDING         │
│  nn.Embedding(5, 8)         │
│                             │
│  Look up row 3 → "i"        │
│  Look up row 4 → "love"     │
│  ...                        │
│                             │
│  (7,) → (7, 8)              │
└─────────────────────────────┘
        │
        ▼
┌─────────────────────────────┐
│   POSITIONAL ENCODING       │
│   (fixed sin/cos table)     │
│                             │
│   pe[:7] shape: (7, 8)      │
│                             │
│   final = token_emb + pe    │
│   (7, 8) + (7, 8) = (7, 8)  │
└─────────────────────────────┘
        │
        ▼
════════════════════════════════
   TRANSFORMER BLOCK STARTS
════════════════════════════════
        │
        ├────────────────────────────────────────┐
        │                                        │ RESIDUAL
        ▼                                        │ (save original)
┌─────────────────────────────┐                  │
│      LAYER NORM 1           │                  │
│   norm1(x)                  │                  │
│                             │                  │
│   Each token's 8 numbers    │                  │
│   rescaled to mean≈0, var≈1 │                  │
│                             │                  │
│   (7, 8) → (7, 8)           │                  │
└─────────────────────────────┘                  │
        │                                        │
        ▼                                        │
┌─────────────────────────────┐                  │
│   MULTI-HEAD ATTENTION      │                  │
│                             │                  │
│  ┌──────────────────────┐   │                  │
│  │ W_Q, W_K, W_V        │   │                  │
│  │ nn.Linear(8, 8)      │   │                  │
│  │                      │   │                  │
│  │ Q = x @ W_Q          │   │                  │
│  │ K = x @ W_K          │   │                  │
│  │ V = x @ W_V          │   │                  │
│  │ (7,8)→(7,8) each     │   │                  │
│  └──────────────────────┘   │                  │
│            │                │                  │
│            ▼                │                  │
│  ┌──────────────────────┐   │                  │
│  │ SPLIT INTO 2 HEADS   │   │                  │
│  │                      │   │                  │
│  │ (7,8) → (7,2,4)      │   │                  │
│  │      → (2,7,4)       │   │                  │
│  │                      │   │                  │
│  │ Head1: (7,4) queries │   │                  │
│  │ Head2: (7,4) queries │   │                  │
│  └──────────────────────┘   │                  │
│            │                │                  │
│            ▼                │                  │
│  ┌──────────────────────┐   │                  │
│  │ ATTENTION per head   │   │                  │
│  │                      │   │                  │
│  │ scores = Q @ K.T     │   │                  │
│  │ (2,7,4)@(2,4,7)      │   │                  │
│  │ = (2,7,7)            │   │                  │
│  │                      │   │                  │
│  │ ÷ √d_k               │   │                  │
│  │                      │   │                  │
│  │ softmax → (2,7,7)    │   │                  │
│  │                      │   │                  │
│  │ @ V → (2,7,4)        │   │                  │
│  └──────────────────────┘   │                  │
│            │                │                  │
│            ▼                │                  │
│  ┌──────────────────────┐   │                  │
│  │ CONCATENATE HEADS    │   │                  │
│  │                      │   │                  │
│  │ (2,7,4) → (7,2,4)    │   │                  │
│  │        → (7,8)       │   │                  │
│  └──────────────────────┘   │                  │
│            │                │                  │
│            ▼                │                  │
│  ┌──────────────────────┐   │                  │
│  │ OUTPUT PROJECTION    │   │                  │
│  │ W_O: nn.Linear(8,8)  │   │                  │
│  │ (7,8) → (7,8)        │   │                  │
│  └──────────────────────┘   │                  │
│                             │                  │
│   output shape: (7, 8)      │                  │
└─────────────────────────────┘                  │
        │                                        │
        ▼                                        │
        + ◄──────────────────────────────────────┘
        │  ADD ORIGINAL INPUT BACK (residual)
        │  (7,8) + (7,8) = (7,8)
        │
        ├────────────────────────────────────────┐
        │                                        │ RESIDUAL
        ▼                                        │ (save this x)
┌─────────────────────────────┐                  │
│      LAYER NORM 2           │                  │
│   norm2(x)                  │                  │
│   (7, 8) → (7, 8)           │                  │
└─────────────────────────────┘                  │
        │                                        │
        ▼                                        │
┌─────────────────────────────┐                  │
│    FEED FORWARD NETWORK     │                  │
│                             │                  │
│  layer1: nn.Linear(8, 32)   │                  │
│  (7,8) → (7,32)   [expand]  │                  │
│                             │                  │
│  ReLU()                     │                  │
│  (7,32) → (7,32)            │                  │
│  [negatives → 0]            │                  │
│                             │                  │
│  layer2: nn.Linear(32, 8)   │                  │
│  (7,32) → (7,8)  [compress] │                  │
│                             │                  │
│  output shape: (7, 8)       │                  │
└─────────────────────────────┘                  │
        │                                        │
        ▼                                        │
        + ◄──────────────────────────────────────┘
        │  ADD INPUT BACK AGAIN (residual)
        │  (7,8) + (7,8) = (7,8)
        │
        ▼
════════════════════════════════
   TRANSFORMER BLOCK OUTPUT
   shape: (7, 8)
════════════════════════════════

Input shape:  (7, 8)
Output shape: (7, 8)  ← same! ready for next block
```